# Assignment 10: Parallel Computing

### Due 19 November 2025

### Introduction

This assignment is about parallel computing with Dask. You should use Python to implement the calculations. If possible, please submit your answers in PDF  or HTML format. In case you have any issues installing Dask via pip, please use the following command:

```bash
python -m pip install "dask[complete]" --use-deprecated=legacy-resolver
```

This command will resolve dependencies and install the required packages for Dask.

You can also install Dask using conda (which the authors recommend):

```bash
conda install dask
```

If you encounter any issues, please check their website: <https://docs.dask.org/en/stable/install.html> and let us know.

1. Explain the concept of "overhead" in parallel computing with `joblib`. Why might running a very simple task (like adding 1 to a number) in parallel with `joblib` be slower than running it serially?

#### Answer:
In `joblib` parallel computing, **overhead** is the extra work required to create worker processes, send data between them, and collect results. For very simple tasks, this overhead is larger than the computation itself. As a result, running tiny tasks in parallel becomes slower than running them serially.


2. Write a Python function `count_vowels(text)` that counts the vowels (a, e, i, o, u, case-insensitive) in a given string. Then, use the `Parallel` and `delayed` functions from the `joblib` library to apply your function in parallel. Use all available cores.  The function should return a list of integers, where each integer corresponds to the number of vowels in the respective sentence.

```python
sentences = [
    "Joblib makes parallel computing easy",
    "Dask scales Python code effectively",
    "Parallelism can speed up computations",
    "Always consider the overhead"
]
```

In [1]:
## Your answer here
from joblib import Parallel, delayed

# Function to count vowels in a sentence
def count_vowels(text):
    vowels = "aeiouAEIOU"
    return sum(1 for char in text if char in vowels)

sentences = [
    "Joblib makes parallel computing easy",
    "Dask scales Python code effectively",
    "Parallelism can speed up computations",
    "Always consider the overhead"
]

# Parallel execution using all available cores
results = Parallel(n_jobs=-1)(
    delayed(count_vowels)(s) for s in sentences
)

print(results)


[12, 10, 13, 10]


3. Write a function called `get_length` that takes a word as input and returns its length. Then, using the provided list `words`, do the following:

* Use a standard (sequential) for loop to calculate the length of each word by calling your function.
* Use the `joblib` library to calculate the length of each word in parallel, also calling your function. Use `Parallel` and `delayed` from `joblib` again.
* Compare the syntax of the sequential and parallel approaches. How do they differ when writing the loop?

```python
words = ["joblib", "parallel", "computing", "example"]
```

In [2]:
## Your answer here
from joblib import Parallel, delayed

# Function to return length of a word
def get_length(word):
    return len(word)

words = ["joblib", "parallel", "computing", "example"]

# 1. Sequential loop
sequential_results = []
for w in words:
    sequential_results.append(get_length(w))

print("Sequential:", sequential_results)

# 2. Parallel loop using joblib
parallel_results = Parallel(n_jobs=-1)(
    delayed(get_length)(w) for w in words
)

print("Parallel:", parallel_results)

# 3. Syntax comparison 
print(
    "Sequential uses a normal for-loop, executing one item at a time.\n"
    "Parallel uses a generator inside Parallel(...), where each function call\n"
    "is wrapped with delayed() to schedule it across multiple workers."
)


Sequential: [6, 8, 9, 7]
Parallel: [6, 8, 9, 7]
Sequential uses a normal for-loop, executing one item at a time.
Parallel uses a generator inside Parallel(...), where each function call
is wrapped with delayed() to schedule it across multiple workers.


3. Create a 10000x10000 Dask array `da_a` filled with random integers between 0 and 100, chunked into (500, 1000) blocks. Use `RandomState(350)` to make your code reproducible. Create a second Dask array `da_b` of the same shape and chunks, filled with ones. Compute `da_c = (da_a + da_b) * 2` and its mean value.

In [6]:
## Your answer here
import dask.array as da

# 1. Create da_a: 10000×10000 random integers (0–100), chunks (500, 1000)
rs = da.random.RandomState(350)
da_a = rs.randint(0, 101, size=(10000, 10000), chunks=(500, 1000))

# 2. Create da_b: same shape, same chunks, filled with ones
da_b = da.ones((10000, 10000), chunks=(500, 1000))

# 3. Compute da_c = (da_a + da_b) * 2
da_c = (da_a + da_b) * 2

# 4. Compute the mean value
mean_value = da_c.mean().compute()

mean_value


101.99377338

4. What is the difference between `dask.dataframe.compute()` and `dask.dataframe.persist()`? When would you typically use `.persist()`?

#### Answer:
`dask.dataframe.compute()` runs all computations immediately and returns the final in-memory result (usually a pandas object).

`dask.dataframe.persist()` starts the computation but keeps the result as a Dask object stored in distributed memory, allowing further operations without recomputing from scratch.

You typically use `.persist()` when you plan to run multiple future computations on the same intermediate result and want to avoid repeating the expensive work.

5. In this question, you will compare the performance of a regular `for` loop and `dask` for a simple computation. First, create a function called `intensive_task` as follows:

```python
import numpy as np
import time
import dask

def intensive_task(n):
    loop_limit = 10_000_000 # How many iterations inside the function
    total = 0
    for i in range(loop_limit):
        total += i*i
    return total
```

Then, create a list called `inputs` with 6 values:

```python
inputs = [1, 2, 3, 4, 5, 6] 
```

Now, use the function `time.time()` to measure the time it takes to run the function `intensive_task` for each value in the list `inputs` using a regular `for` loop. Store the results in a list called `results`. Remember to create the `start_time` and `end_time` variables to measure the time taken for the computation. The result, which is the difference between `end_time` and `start_time`, should be printed.

Repeat the same task using `dask`. However, instead of using the `@dask.delayed` decorator, use the code below:

```python
tasks = [dask.delayed(intensive_task)(i) for i in inputs]
```

Then, use `dask.compute()` to compute the results. Again, measure the time taken for the computation and print the result. Which one is faster?

In [7]:
## Your answer here
import numpy as np
import time
import dask

# The given intensive function
def intensive_task(n):
    loop_limit = 10_000_000
    total = 0
    for i in range(loop_limit):
        total += i * i
    return total

# 1. Serial computation using a normal for-loop

inputs = [1, 2, 3, 4, 5, 6]

start_time = time.time()

results_serial = []
for i in inputs:
    results_serial.append(intensive_task(i))

end_time = time.time()
serial_time = end_time - start_time

print("Serial results:", results_serial)
print("Serial time:", serial_time, "seconds")

# 2. Parallel computation using dask.delayed

tasks = [dask.delayed(intensive_task)(i) for i in inputs]

start_time = time.time()
results_dask = dask.compute(*tasks)
end_time = time.time()

dask_time = end_time - start_time

print("Dask results:", results_dask)
print("Dask time:", dask_time, "seconds")


# 3. Which one is faster?
if dask_time < serial_time:
    print("Dask is faster.")
else:
    print("Serial for-loop is faster.")


Serial results: [333333283333335000000, 333333283333335000000, 333333283333335000000, 333333283333335000000, 333333283333335000000, 333333283333335000000]
Serial time: 2.0639078617095947 seconds
Dask results: (333333283333335000000, 333333283333335000000, 333333283333335000000, 333333283333335000000, 333333283333335000000, 333333283333335000000)
Dask time: 1.9818711280822754 seconds
Dask is faster.


6. In the same folder as this notebook, you will find a Parquet file named `data.parquet`. It is available here: <https://github.com/danilofreire/qtm350/blob/main/assignments/data.parquet>. This file contains student records with the following columns:

* `emory_id` (integer) 
* `student_name` (string)
* `major` (string)
* `gpa` (float)

Write Python code using `dask.dataframe` to read the `data.parquet` file, but only load the `major` and `gpa` columns. Then, print the first 5 rows of the resulting Dask DataFrame using the `.head()` method, and calculate the average GPA by major.

You will need a Parquet engine to read the file. If you don't have one installed, you can use `pyarrow`. You can install it using conda (or pip):

```bash
conda install pyarrow
```

In [8]:
## Your answer here
import dask.dataframe as dd

# Load only the 'major' and 'gpa' columns from the parquet file
df = dd.read_parquet("data.parquet", columns=["major", "gpa"])

# Print first 5 rows
print(df.head())

# Calculate average GPA by major
avg_gpa = df.groupby("major")["gpa"].mean().compute()

print("\nAverage GPA by major:")
print(avg_gpa)


       major   gpa
0    History  2.98
1  Chemistry  3.16
2  Chemistry  3.83
3        QTM  3.67
4    CompSci  3.33

Average GPA by major:
major
Biology      3.044000
Chemistry    3.320000
CompSci      3.352857
Economics    3.285000
English      3.484000
History      3.098750
Physics      3.222857
QTM          2.957500
Name: gpa, dtype: float64


7. You have two CSV files in this directory:

* `students.csv`: Contains columns `student_id`, `student_name`. Available here: <https://github.com/danilofreire/qtm350/blob/main/assignments/students.csv>.
* `grades.csv`: Contains columns `student_id`, `course`, `grade`. Available here: <https://github.com/danilofreire/qtm350/blob/main/assignments/grades.csv>.

Write Python code using dask.dataframe to:

* Read `students.csv` into a Dask DataFrame called `ddf_students`.
* Read `grades.csv` into a Dask DataFrame called `ddf_grades`.
* Merge these two DataFrames together based on the common `student_id` column. An inner merge is recommended (only include students present in both files).
* From the merged DataFrame, select only the `student_name`, `course`, and `grade` columns. Save it as `ddf_final`.
* Compute and print the first 5 rows of this final merged DataFrame using `.head()`.

In [9]:
import dask.dataframe as dd

# 1. Read CSV files into Dask DataFrames
ddf_students = dd.read_csv("students.csv")
ddf_grades   = dd.read_csv("grades.csv")

# 2. Merge on the common column: student_id (inner join)
ddf_merged = ddf_students.merge(ddf_grades, on="student_id", how="inner")

# 3. Select only the required columns
ddf_final = ddf_merged[["student_name", "course", "grade"]]

# 4. Compute and print the first 5 rows
print(ddf_final.head())


  student_name  course grade
0        Alice  QTM100     A
1        Alice  QTM200     B
2          Bob  QTM100     B
3          Bob  QTM300     C
4      Charlie  QTM100     A


Good luck! 😃